# Exercise 2 — Build and Calibrate an LLM-as-Judge

In Exercise 1 you read the QA bot's answers yourself and judged them by
eye. That doesn't scale — you can't manually read thousands of answers
every time you change a prompt. Here you'll build an **automated judge**:
a second model call that grades a (question, answer) pair and returns a
structured pass/fail verdict.

The hard part isn't calling the model — it's writing a rubric precise
enough that the judge agrees with a human most of the time. You'll write
a rubric, run it against 8 pre-labeled examples, measure how often it
agrees with the human labels, and revise the rubric where it doesn't.

## Setup

Needs your own Anthropic API key, and the repo cloned so `judge.py` and
the shared fixtures are available.

In [ ]:
!pip install -q anthropic
%cd /content
!rm -rf repo
!git clone -q https://github.com/PrepVector/Agent_Evals.git repo
%cd repo/workshop_demo_claude/judge

import os, json
# Replace PASTE_KEY_HERE with your API key. Do not share or commit this notebook with your key in it.
os.environ["ANTHROPIC_API_KEY"] = "PASTE_KEY_HERE"
assert os.environ["ANTHROPIC_API_KEY"] != "PASTE_KEY_HERE", "Replace PASTE_KEY_HERE with your API key first."

from judge import judge_answer

with open("labeled_examples.json") as f:
    LABELED_EXAMPLES = json.load(f)

print(f"Loaded {len(LABELED_EXAMPLES)} labeled examples.")
print("Human verdicts:", {v: sum(1 for e in LABELED_EXAMPLES if e["human_verdict"] == v) for v in ("pass", "fail")})

## 1. Read the labeled examples first

These are real (question, answer) pairs a QA bot might produce, each
already graded by a human with a one-line reason why. Read through all
8 before writing your rubric -- you're trying to write criteria general
enough to reproduce *all* of these judgments, not just the first one.

In [ ]:
for ex in LABELED_EXAMPLES:
    print(f"[{ex['human_verdict'].upper()}] {ex['id']}")
    print(f"  Q: {ex['question']}")
    print(f"  A: {ex['answer']}")
    print(f"  Why: {ex['human_reasoning']}")
    print()

## 2. Write your rubric

Two criteria are filled in as a starting point. **Add at least 3 more**
based on what you saw in the labeled examples above -- look especially
at the `fail` cases and ask "what general rule would this violate?"

The rubric is plain text -- the judge will read it as grading
instructions, so write it as instructions, not just keywords.

In [ ]:
# YOUR TURN: add at least 3 more criteria inside the quotes below.
# Ideas to get you started (these comments are NOT sent to the judge):
#   - a criterion about citing sources
#   - a criterion about what the answer should do when the context doesn't cover the question
#   - a criterion about fully answering multi-part / compound questions
RUBRIC = """
- The answer must be grounded only in the provided context documents (no outside knowledge, no guessing).
- If multiple documents give conflicting information on the same fact, the answer should use the most recent one.
"""

print(RUBRIC)

## 3. Run your judge against all 8 examples

This calls `judge_answer()` once per example and compares its verdict
to the human label.

In [ ]:
results = []
for ex in LABELED_EXAMPLES:
    verdict = judge_answer(ex["question"], ex["answer"], RUBRIC)
    agree = verdict["verdict"] == ex["human_verdict"]
    results.append({**ex, "judge_verdict": verdict["verdict"], "judge_reasoning": verdict["reasoning"], "agree": agree})

    mark = "MATCH" if agree else "DISAGREE"
    print(f"[{mark}] {ex['id']}  human={ex['human_verdict']}  judge={verdict['verdict']}")
    if not agree:
        print(f"    human said:  {ex['human_reasoning']}")
        print(f"    judge said:  {verdict['reasoning']}")
    print()

agreement_rate = sum(r["agree"] for r in results) / len(results)
print(f"Agreement with human labels: {agreement_rate:.0%} ({sum(r['agree'] for r in results)}/{len(results)})")

## 4. Iterate

For every example your judge disagreed on: read the judge's reasoning
next to the human's. Is the judge wrong, or did it catch something the
rubric didn't ask it to check that the human *implicitly* expected?

Revise `RUBRIC` above to close the gap, then **re-run the two cells
above**. Repeat until you're at 100% agreement, or until you're confident
the remaining disagreement is a genuinely ambiguous case rather than a
missing rubric criterion.

## 5. Reflection

- What kind of rubric criteria were easy for the judge to apply
  consistently, and which ones needed several rewrites?
- If you handed your final rubric to someone else in the room, do you
  think their judge would produce the same verdicts as yours? What
  would make you more confident either way?
- This judge only sees one (question, answer) pair at a time. What
  would you need to change to use a judge like this on an **agent
  trajectory** instead of a single answer -- grading the sequence of
  steps, not just the final text? (This is exactly what Section 6's
  trajectory judge would need to do differently.)